In [1]:
!pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 23.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 9.2 MB/s eta 0:00:00


In [3]:
import os
import mlflow
from kaggle_secrets import UserSecretsClient

DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("DAGSHUB_TOKEN")

mlflow.set_tracking_uri("https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow")
mlflow.set_experiment("RandomForest_Training")

2026/05/03 16:13:00 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/c3684fd113ea4232be3099692ca85877', creation_time=1777824780240, experiment_id='2', last_update_time=1777824780240, lifecycle_stage='active', name='RandomForest_Training', tags={}, trace_location=None, workspace='default'>

In [4]:
import pandas as pd

train_t = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_i = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
test_t  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_i  = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

print("train_transaction:", train_t.shape)
print("train_identity:   ", train_i.shape)
print("test_transaction: ", test_t.shape)
print("test_identity:    ", test_i.shape)

train_transaction: (590540, 394)
train_identity:    (144233, 41)
test_transaction:  (506691, 393)
test_identity:     (141907, 41)


In [5]:
train = train_t.merge(train_i, how="left", on="TransactionID")
test  = test_t.merge(test_i,  how="left", on="TransactionID")

print("train merged:", train.shape)
print("test merged: ", test.shape)
print()
print("Fraud distribution:")
print(train["isFraud"].value_counts())
print("Fraud rate:", train["isFraud"].mean())

train merged: (590540, 434)
test merged:  (506691, 433)

Fraud distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64
Fraud rate: 0.03499000914417313


Cleaning

In [6]:
nan_pct = train.isna().mean().sort_values(ascending=False)
print(nan_pct.head(20))
print()
print("columns with >90% missing:", (nan_pct > 0.9).sum())
print("columns with >50% missing:", (nan_pct > 0.5).sum())

id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_27    0.991247
id_23    0.991247
id_22    0.991247
dist2    0.936284
D7       0.934099
id_18    0.923607
D13      0.895093
D14      0.894695
D12      0.890410
id_04    0.887689
id_03    0.887689
D6       0.876068
id_33    0.875895
id_09    0.873123
dtype: float64

columns with >90% missing: 12
columns with >50% missing: 214


In [7]:
test = test.rename(columns={c: c.replace("-", "_") for c in test.columns})
print("test cols renamed")

test cols renamed


In [8]:
threshold = 0.9
drop_cols = nan_pct[nan_pct > threshold].index.tolist()
with mlflow.start_run(run_name="RandomForest_Cleaning_t09"):
    mlflow.log_param("nan_threshold", threshold)
    mlflow.log_param("dropped_cols", len(drop_cols))
    train_c = train.drop(columns=drop_cols)
    test_c = test.drop(columns=drop_cols)
    mlflow.log_metric("cols_after", train_c.shape[1])
print("dropped:", len(drop_cols))
print("shape after:", train_c.shape)


🏃 View run RandomForest_Cleaning_t09 at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/f0017b50d3854d2d91c1479496a69936
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
dropped: 12
shape after: (590540, 422)


Feature Engineering

In [9]:
train_c["TransactionAmt"].describe()

count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

In [10]:
import numpy as np
train_c["hour"] = (train_c["TransactionDT"] // 3600) % 24
test_c["hour"] = (test_c["TransactionDT"] // 3600) % 24
train_c["day"] = (train_c["TransactionDT"] // (3600*24)) % 7
test_c["day"] = (test_c["TransactionDT"] // (3600*24)) % 7
print(train_c[["TransactionAmt", "hour", "day"]].head())

   TransactionAmt  hour  day
0            68.5     0    1
1            29.0     0    1
2            59.0     0    1
3            50.0     0    1
4            50.0     0    1


In [11]:
with mlflow.start_run(run_name="RandomForest_Feature_Engineering"):
    mlflow.log_param("time_features", "hour,day")
    mlflow.log_metric("new_features", 2)
    mlflow.log_metric("cols_total", train_c.shape[1])
print("cols total:", train_c.shape[1])

🏃 View run RandomForest_Feature_Engineering at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/b4774b089f9c4beeb3ff472f3d16438d
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
cols total: 424


Feature Selection

In [12]:
threshold_corr = 0.95
num_cols = train_c.select_dtypes(include=[np.number]).columns.drop(["isFraud", "TransactionID"])
corr = train_c[num_cols].sample(100000, random_state=42).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > threshold_corr).any()]
with mlflow.start_run(run_name="RandomForest_Feature_Selection"):
    mlflow.log_param("corr_threshold", threshold_corr)
    mlflow.log_param("dropped_corr", len(drop_corr))
    train_fs = train_c.drop(columns=drop_corr)
    test_fs = test_c.drop(columns=drop_corr)
    mlflow.log_metric("cols_after_fs", train_fs.shape[1])
print("dropped:", len(drop_corr))
print("shape after FS:", train_fs.shape)

🏃 View run RandomForest_Feature_Selection at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/3e2a9aa0778e434d90015a26a298f6fb
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
dropped: 139
shape after FS: (590540, 285)


In [13]:
from sklearn.ensemble import RandomForestClassifier
sample = train_fs.sample(100000, random_state=42)
y = sample["isFraud"]
X = sample.drop(columns=["isFraud", "TransactionID"]).copy()
for c in X.select_dtypes(include=["object"]).columns:
    X[c] = X[c].fillna("missing").astype("category").cat.codes
X = X.fillna(-999)
quick_rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
quick_rf.fit(X, y)
imp = pd.Series(quick_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top_cols = imp.head(100).index.tolist()
drop_low_imp = [c for c in X.columns if c not in top_cols]
train_fs = train_fs.drop(columns=drop_low_imp)
test_fs = test_fs.drop(columns=drop_low_imp)
with mlflow.start_run(run_name="RandomForest_Feature_Selection_importance"):
    mlflow.log_param("top_n_features", 100)
    mlflow.log_param("dropped_low_imp", len(drop_low_imp))
    mlflow.log_metric("cols_after_imp", train_fs.shape[1])
print("dropped:", len(drop_low_imp))
print("shape after:", train_fs.shape)

🏃 View run RandomForest_Feature_Selection_importance at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/b6b971beac714cb495659fdeddfab3ef
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
dropped: 183
shape after: (590540, 102)


Training

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin
class FraudPrep(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
    def fit(self, X, y=None):
        self.cat_maps_ = {}
        self.medians_ = {}
        Xt = self._prep(X.drop(columns=[c for c in self.drop_cols if c in X.columns]))
        for c in Xt.columns:
            if Xt[c].dtype == "object":
                cats = Xt[c].fillna("missing").astype(str).unique().tolist()
                self.cat_maps_[c] = {v: i for i, v in enumerate(cats)}
            else:
                self.medians_[c] = Xt[c].median()
        return self
    def transform(self, X):
        Xt = self._prep(X.drop(columns=[c for c in self.drop_cols if c in X.columns]))
        for c, m in self.cat_maps_.items():
            if c in Xt.columns:
                Xt[c] = Xt[c].fillna("missing").astype(str).map(m).fillna(-1).astype(int)
        for c, med in self.medians_.items():
            if c in Xt.columns:
                Xt[c] = Xt[c].fillna(med)
        return Xt.values
    def _prep(self, X):
        X = X.copy()
        X["hour"] = (X["TransactionDT"] // 3600) % 24
        X["day"] = (X["TransactionDT"] // (3600*24)) % 7
        return X


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
sample = train_fs.sample(100000, random_state=42)
y = sample["isFraud"]
X = sample.drop(columns=["isFraud", "TransactionID"]).copy()
for c in X.select_dtypes(include=["object"]).columns:
    X[c] = X[c].fillna("missing").astype("category").cat.codes
X = X.fillna(-999)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for d in [5, 15, None]:
    for cw in [None, "balanced"]:
        m = RandomForestClassifier(n_estimators=100, max_depth=d, class_weight=cw, n_jobs=-1, random_state=42)
        scores = cross_val_score(m, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
        cw_label = "none" if cw is None else "balanced"
        with mlflow.start_run(run_name=f"RandomForest_Training_d{d}_{cw_label}"):
            mlflow.log_param("n_estimators", 100)
            mlflow.log_param("max_depth", str(d))
            mlflow.log_param("class_weight", cw_label)
            mlflow.log_param("cv_folds", 3)
            mlflow.log_param("sample_size", len(X))
            mlflow.log_metric("cv_auc_mean", scores.mean())
            mlflow.log_metric("cv_auc_std", scores.std())
        print(f"d={d}, cw={cw_label}: AUC={scores.mean():.4f} +/- {scores.std():.4f}")

🏃 View run RandomForest_Training_d5_none at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/2f6192ffc9cd460e83f909e6341e2e44
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
d=5, cw=none: AUC=0.8340 +/- 0.0040
🏃 View run RandomForest_Training_d5_balanced at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/cf8530f86f6f4d7da89852c8275ab5ff
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
d=5, cw=balanced: AUC=0.8459 +/- 0.0018
🏃 View run RandomForest_Training_d15_none at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/a5bf3d1f41b8417d9ce418c001a93d8a
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
d=15, cw=none: AUC=0.8748 +/- 0.0009
🏃 View run RandomForest_Training_d15_balanced at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#

In [16]:
from sklearn.pipeline import Pipeline
nan_pct = train.isna().mean()
nan_drop = nan_pct[nan_pct > 0.9].index.tolist()
all_drop = list(set(nan_drop) | set(drop_corr) | set(drop_low_imp) | {"TransactionID"})
y_full = train["isFraud"]
X_full = train.drop(columns=["isFraud"])
final_pipe = Pipeline([("prep", FraudPrep(drop_cols=all_drop)), ("rf", RandomForestClassifier(n_estimators=100, max_depth=None, class_weight="balanced", n_jobs=-1, random_state=42))])
final_pipe.fit(X_full, y_full)
print("fitted")

fitted


In [17]:
import joblib
joblib.dump(final_pipe, "rf_pipeline.pkl")
test_preds = final_pipe.predict_proba(test)[:, 1]
print("preds shape:", test_preds.shape, "first:", test_preds[:3])
with mlflow.start_run(run_name="RandomForest_FinalPipeline"):
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", "None")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("cv_auc", 0.8960)
    mlflow.log_artifact("rf_pipeline.pkl")
print("saved")

preds shape: (506691,) first: [0.02 0.   0.03]
🏃 View run RandomForest_FinalPipeline at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2/runs/9b83cbc838a7408eb65f983ab12a67a8
🧪 View experiment at: https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow/#/experiments/2
saved
